In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Top 10 equipos con más goles anotados como local
goles_local = (
    df.groupby("equipo_local")["goles_local"]
    .sum()
    .reset_index()
    .rename(columns={"equipo_local": "equipo", "goles_local": "total_goles"})
    .sort_values("total_goles", ascending=False)
    .head(10)
)

# Gráfico
plt.figure(figsize=(12, 6))
ax = sns.barplot(
    data=goles_local,
    x="equipo",
    y="total_goles",
    palette="Blues_d",
    order=goles_local["equipo"]
)

# Etiquetas encima de cada barra
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", padding=3, fontsize=10)

ax.set_title("Top 10 equipos con más goles como local", fontsize=15, fontweight="bold", pad=15)
ax.set_xlabel("Equipo", fontsize=12)
ax.set_ylabel("Total de goles", fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# Tabla pivote: cantidad de partidos jugados entre equipos
heatmap_data = (
    df.groupby(["equipo_local", "equipo_visitante"])
    .size()
    .reset_index(name="partidos")
    .pivot(index="equipo_local", columns="equipo_visitante", values="partidos")
    .fillna(0)
    .astype(int)
)

# Tamaño dinámico según número de equipos
n = len(heatmap_data)
fig_size = max(12, n * 0.6)

# Heatmap
plt.figure(figsize=(fig_size, fig_size * 0.8))
ax = sns.heatmap(
    heatmap_data,
    annot=True,
    fmt="d",
    cmap="YlOrRd",
    linewidths=0.4,
    linecolor="white",
    cbar_kws={"label": "Nº de partidos", "shrink": 0.6},
    annot_kws={"size": 8}
)

ax.set_title("Cantidad de partidos jugados entre equipos\n(Local → filas | Visitante → columnas)",
             fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Equipo visitante", fontsize=11)
ax.set_ylabel("Equipo local", fontsize=11)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

plt.tight_layout()
plt.show()

## 📝 Conclusiones

A partir de los gráficos generados en este análisis, se pueden destacar las siguientes observaciones:

1. **Ventaja de localía en la anotación:** Los equipos históricamente fuertes concentran la mayor cantidad de goles anotados como local, lo que evidencia que jugar en casa representa una ventaja competitiva clara y consistente en la Premier League.

2. **Brecha entre rendimiento local y visitante:** Al comparar los promedios de goles, prácticamente todos los equipos muestran un mayor rendimiento ofensivo cuando juegan como locales frente a cuando juegan como visitantes, confirmando el conocido efecto de localía en el fútbol profesional.

3. **Distribución equilibrada de enfrentamientos:** El heatmap revela que la cantidad de partidos jugados entre equipos es homogénea, lo cual es consistente con el formato de liga donde cada equipo se enfrenta a los demás el mismo número de veces, tanto en casa como fuera.

In [ ]:
import plotly.graph_objects as go

# Promedios de goles por equipo (local y visitante)
prom_local = (
    df.groupby("equipo_local")["goles_local"]
    .mean()
    .reset_index()
    .rename(columns={"equipo_local": "equipo", "goles_local": "prom_local"})
)

prom_visitante = (
    df.groupby("equipo_visitante")["goles_visitante"]
    .mean()
    .reset_index()
    .rename(columns={"equipo_visitante": "equipo", "goles_visitante": "prom_visitante"})
)

# Merge y ordenar por promedio local descendente
prom = prom_local.merge(prom_visitante, on="equipo").sort_values("prom_local", ascending=False)

# Gráfico interactivo
fig = go.Figure()

fig.add_trace(go.Bar(
    name="Goles local (prom.)",
    x=prom["equipo"],
    y=prom["prom_local"].round(2),
    marker_color="#1f77b4",
    text=prom["prom_local"].round(2),
    textposition="outside"
))

fig.add_trace(go.Bar(
    name="Goles visitante (prom.)",
    x=prom["equipo"],
    y=prom["prom_visitante"].round(2),
    marker_color="#ff7f0e",
    text=prom["prom_visitante"].round(2),
    textposition="outside"
))

fig.update_layout(
    title=dict(text="Promedio de goles: local vs visitante por equipo", font=dict(size=16)),
    barmode="group",
    xaxis=dict(title="Equipo", tickangle=-35),
    yaxis=dict(title="Promedio de goles por partido"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    template="plotly_white",
    height=520
)

fig.show()

## Visualizaciones - Premier League 2024
En este notebook se presentan gráficos interactivos y estadísticos para analizar los datos de la Premier League. Se exploran los equipos con más goles, comparativas local vs visitante y patrones entre equipos.

In [ ]:
import pandas as pd
import sqlite3
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

conn = sqlite3.connect("data/premier_league.db")
df = pd.read_sql("SELECT * FROM partidos", conn)
conn.close()

print(df.shape)
df.head()